In [ ]:
import kagglehub

path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')

print("Path to competition files:", path)

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
import cv2
%matplotlib inline

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# TorchVision
import torchvision.transforms.v2 as v2
from torchvision.models import resnet50, densenet121, ResNet50_Weights, DenseNet121_Weights,swin_v2_b,Swin_V2_B_Weights
from torchvision.utils import make_grid

# Data utilities
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Evaluation
from sklearn.metrics import confusion_matrix, classification_report,roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
MAIN_PATH = "/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data"
TRAIN_PATH = os.path.join(MAIN_PATH, 'Training')
TEST_PATH = os.path.join(MAIN_PATH, 'Test')
TRAIN_CSV_PATH = os.path.join(MAIN_PATH, 'training.csv')
TEST_CSV_PATH = os.path.join(MAIN_PATH, 'test.csv')
LABEL_TO_NAMES_PATH = os.path.join(MAIN_PATH, 'sources.txt')

NUM_OF_CLASSES = 10

BATCH_SIZE = 64
EPOCHS = 10
PATIENCE = 5
LR = 1e-4

MEAN_NORM = [0.485, 0.456, 0.406]
STD_NORM  = [0.229, 0.224, 0.225]

NUM_WORKERS = 4
PIN_MEMORY = True

In [3]:
class SynthiticDataset(Dataset):
    def __init__(self,df_path,transform = None):
        self.df = pd.read_csv(df_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        image_path = self.df.iloc[idx,1]
        label = self.df.iloc[idx,2]

        img = Image.open(image_path).convert('RGB')

        if self.transform:
            self.transform(img)

        return img,label
        